# Introduction: Titanic Exercise

At the start of the semester, the well-known Titanic exercise was introduced as an
introductory assignment to begin exploring the field of artificial intelligence and
data analysis. The purpose of this exercise is to gain hands-on experience with basic
AI-related workflows while becoming familiar with common data science tools.

This notebook explores the Titanic dataset using Python and widely used data analysis
libraries. The goal is to examine the data, identify patterns, and answer questions
related to passenger survival, providing a foundation for future work in machine
learning and artificial intelligence.


## Step 1: Explore the dataset

As I have a programming background, I used the Visual Studio Code environment and the provided Python notebook to work on this exercise.

Various criteria must be met before this step can be considered complete.
How these criteria are met is explained below.

### Criteria
- Load the dataset into a table (a DataFrame).
- Show the first rows and the column names.
- Identify missing values (which columns have gaps?).
- Answer the following questions:
  - What percentage survived?
  - Did survival differ by ticket class?
  - Did survival differ by gender?
  - Does age seem to matter?
  - Does ticket price relate to survival?



The first two criteria are straightforward and are completed in three lines, as shown below.


In [ ]:
import sklearn
import pandas
import seaborn

data = pandas.read_csv("phase0-titanic/titanic.csv")
data.sample(5)

For the third criterion, a single line gives us the missing values.
The output shows that there should be 1,313 entries per column. We can see that many values are missing.


In [ ]:
data.info()

For the final criterion, several questions must be answered.

To answer most of these questions, it is important to know how many passengers survived. This gives us a baseline.


In [ ]:
data['survived'].value_counts()


While this gives a quick overview with very few lines of code, there is an even quicker way to return the values as percentages.


In [ ]:
nu_survived = (data['survived'] == 1).sum()
pr_survived = (data['survived'].mean() * 100).round(2)

print(f"Total survived : {nu_survived}")
print(f"Percentage survived : {pr_survived}%")

Now that the survival proportion is established, we can examine whether there are major differences in survival rates between subgroups.


In [ ]:
grouped = data.groupby('pclass')['survived']

for pclass, group in grouped:
    total_passengers = group.count()
    total_survived = group.sum()                 
    percentage_survived = round(group.mean() * 100, 2) 
    
    print(f"Class {pclass}:")
    print(f"  Total in class : {total_passengers}")
    print(f"  Total survived : {total_survived}")
    print(f"  Percentage survived : {percentage_survived}%\n")

In [ ]:
grouped = data.groupby('sex')['survived']

for sex, group in grouped:
    total_passengers = group.count()
    total_survived = group.sum()                  
    percentage_survived = round(group.mean() * 100, 2)  
    
    print(f"Class {sex}:")
    print(f"  Total passengers : {total_passengers}")
    print(f"  Total survived : {total_survived}")
    print(f"  Percentage survived : {percentage_survived}%\n")

For age, it is better to create our own subgroups rather than analyze every specific age.
Generally, we can group people as children (0-17), young adults (18-35), adults (36-60), and elderly people (61+).
These subgroups provide a clear indication of how life stage may influence survival rate.

It is important to note that only about half of the passengers had their age listed. So while the results provide a general idea, they are not fully reliable because of the large amount of missing data.


In [ ]:
bins = [0, 17, 35, 60, 120]
labels = ['Children', 'Young Adult', 'Adult', 'Elderly']

age_grouped = data.groupby(pandas.cut(data['age'], bins=bins, labels=labels))['survived']

for age_group, group in age_grouped:
    total_passengers = group.count()
    total_survived = group.sum()
    percentage_survived = round(group.mean() * 100, 2)
    
    print(f"{age_group}:")
    print(f"  Total in group : {total_passengers}")
    print(f"  Total survived : {total_survived}")
    print(f"  Percentage survived : {percentage_survived}%\n")

Now the question is: can we draw conclusions from this?
If we only look at the available data, we can reasonably say yes.
However, it is not that straightforward because we do not know all circumstances.
Passenger data alone makes it difficult to draw firm conclusions; with more contextual data about the situation, we could interpret these results more confidently.


## Step 2: Building a first model

For the second step, we take a first dive into how to create a model and how to use it.

For this step, several criteria must be met in order to consider it complete.

### Criteria

- Select a few features (for example: Pclass, Sex, Age, Fare, Embarked).
- Handle missing values in a simple way (for example: fill missing ages with the median).
- Train a model (for example: logistic regression, decision tree, or random forest).
- Evaluate the model using a simple metric (accuracy is fine).
- Compare your model to a “stupid baseline” (for example: predicting that nobody survives, or that everyone survives).



To create a model, a new DataFrame is created based on the initial DataFrame.
Some columns are used while others are omitted.
As found earlier, there are many missing values.
These missing values are addressed by using either the average or the most common value, depending on the category.
In addition, most ML algorithms do not work directly with strings or text, so categorical values are mapped to numerical values.


**sex**
| Original | Numeric |
| -------- | ------- |
| female   | 0       |
| male     | 1       |

**pclass**
| Original | Numeric |
| -------- | ------- |
| 1st      | 1       |
| 2nd      | 2       |
| 3rd      | 3       |



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

features = ['pclass', 'sex', 'age', 'embarked']
x = data[features]
y = data['survived']

x['age'] = x['age'].fillna(x['age'].median())
x['embarked'] = x['embarked'].fillna(x['embarked'].mode()[0])

x["pclass"] = x["pclass"].map({"1st":1, "2nd":2, "3rd":3}).astype(int)
x["sex"] = x["sex"].map({"male":0, "female":1}).astype(int)

x = pandas.get_dummies(x, columns=['embarked'], drop_first=False)
x = pandas.get_dummies(x, columns=['pclass'],drop_first=False)

x.sample(5)

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest: {accuracy_rf:.2f}")



In [ ]:
importances = rf_model.feature_importances_
feature_names = x.columns

importances_pct = importances * 100

indices = np.argsort(importances_pct)[::-1]

plt.figure(figsize=(8,5))
bars = plt.bar(
    range(len(importances_pct)),
    importances_pct[indices]
)

plt.xticks(
    range(len(importances_pct)),
    feature_names[indices],
    rotation=45
)

plt.ylabel("Importance (%)")
plt.title("Random Forest Feature Importances Passengers")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report
target_names = ["no", "yes"]
print("Train set:")
predictions = rf_model.predict(X_train)
report = classification_report(y_train, predictions, target_names=target_names)
print(report)

print("Test set:")
predictions = rf_model.predict(X_test)
report = classification_report(y_test, predictions, target_names=target_names)
print(report)

Personally, I am interested in whether there are differences in accuracy between various machine learning models.
Therefore, I decided to compare this model with two other approaches.
As some models are considered stronger than others, I will use cross-validation to produce more reliable accuracy estimates and reduce randomness.

### Different models



In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)

# KNN Model
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)
accuracy_knn = accuracy_score(y_test, y_pred_knn)

#XGboost
xgb_model = XGBClassifier(eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

#Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Model Accuracy Comparison (80/20 Split):")
print(f"Random Forest: {accuracy_rf:.2f}")
print(f"KNN: {accuracy_knn:.2f}")
print(f"XGBoost: {accuracy_xgb:.2f}")
print(f"Logistic Regression: {accuracy_lr:.2f}")


In [ ]:
import matplotlib.pyplot as plt

models = ["Random Forest", "KNN", "XGBoost", "Logistic Regression"]
accuracies = [accuracy_rf, accuracy_knn, accuracy_xgb, accuracy_lr]

plt.figure(figsize=(10, 5))
plt.bar(models, accuracies)
plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison (80/20 Split)")
plt.xticks(rotation=30, ha='right')
plt.ylim(0.6, 1.0)
plt.tight_layout()
plt.show()

# Step 3: Including Crew

I noticed that the given dataset only includes passengers. The Titanic had over 900 crew members who are not accounted for.
Considering this, I was curious whether including crew data would make a difference and, if so, how.
Therefore, I decided to investigate this further with a different dataset from Kaggle.
I will use Random Forest, as this was the initial model used to calculate feature importance and compare results against the passenger-only dataset.

Unlike the first dataset, the dataset that includes crew does not have a separate test set.
Therefore, five-fold cross-validation will be used.



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split

data_full = pd.read_csv("phase0-titanic/titanic_full.csv")

grouped = data_full.groupby('pclass')['survived']

for pclass, group in grouped:
    total_passengers = group.count()
    total_survived = group.sum()                 
    percentage_survived = round(group.mean() * 100, 2)  
    
    print(f"Class {pclass}:")
    print(f"  Total in class : {total_passengers}")
    print(f"  Total survived : {total_survived}")
    print(f"  Percentage survived : {percentage_survived}%\n")
    
columns = ['pclass', 'sex', 'age', 'embarked']
x = data_full[columns].copy()
y = data_full['survived']

x['age'] = x['age'].fillna(x['age'].median())
x['embarked'] = x['embarked'].fillna(x['embarked'].mode()[0])

x['is_crew'] = ~data_full['pclass'].isin(['1st', '2nd', '3rd'])
x['is_crew'] = x['is_crew'].astype(int)
x['sex'] = x['sex'].map({'male': 0, 'female': 1}).astype(int)

x = pd.get_dummies(x, columns=['pclass'], drop_first=False)
x = pd.get_dummies(x, columns=['embarked'], drop_first=False)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
x.drop(columns=['is_crew'], inplace=True)
rf.fit(x, y)
cv_scores = cross_val_score(rf, x, y, cv=5)

print(f"Random Forest 5-fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


## Plot Importance

To compare how feature importance changes when crew members are included, a second plot is created.


In [ ]:
importances = pd.Series(rf.feature_importances_, index=x.columns)
importances = importances.sort_values(ascending=False)
importances_percent = importances * 100

plt.figure(figsize=(12,6))
bars = plt.bar(importances_percent.index, importances_percent.values)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Importance (%)")
plt.title("Random Forest Feature Importances (including crew indicator)")

for bar, pct in zip(bars, importances_percent.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Conclusion

When crew members are included in the survival analysis, the most important features still appear to be sex and age, regardless of whether the dataset contains only passengers or both passengers and crew.
It seems that the relative importance of age increases when crew members are included.
In addition, people with a so-called higher status appear to have a higher survival rate.
This is reflected in the fact that first-class passengers and deck crew both have a survival rate above 65%.


# Reflection

During this assignment, I learned how important data quality is before training a model.
A large part of the work was not model training itself, but understanding the dataset,
checking missing values, and making reasonable preprocessing choices.
I also learned that interpretation matters: patterns such as gender, age, and class can be observed,
but conclusions should be made carefully when data is incomplete or context is limited.

From a machine learning perspective, I learned how to move from exploration to a first predictive model,
and how to evaluate that model against a baseline.
Comparing multiple models with cross-validation helped me understand that model performance can vary,
and that a single train/test split is not always enough for a reliable comparison.
Including crew data also showed me how changing the dataset can influence feature importance and model outcomes.

AI supported me in several ways during this process.
It helped me structure the workflow step by step, improve explanations in the notebook,
and quickly verify whether my reasoning and code logic were clear.
AI was especially useful for refining written analysis, checking language quality,
and generating suggestions for clearer phrasing and stronger argumentation.
At the same time, I learned that AI support is most valuable when I critically review its suggestions
and keep ownership of the final decisions and interpretations.
